In [1]:
import gspread
from google.oauth2.service_account import Credentials
import pandas as pd
import duckdb

print("🔐 Autenticando con Google...")
CONFIG_SCOPES = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive'
]

# Cargar credenciales locales
credenciales = Credentials.from_service_account_file('credenciales.json', scopes=CONFIG_SCOPES)
cliente_gmail = gspread.authorize(credenciales)

# Conectar al archivo exacto
sheet_id = cliente_gmail.open('DB_Tienda_Operacional')
print("✅ Conectado exitosamente a DB_Tienda_Operacional")

🔐 Autenticando con Google...
✅ Conectado exitosamente a DB_Tienda_Operacional


In [2]:
print("📥 1/2: Extrayendo tablas desde Google Sheets...")
df_ventas = pd.DataFrame(sheet_id.worksheet('Ventas').get_all_records())
df_detalle = pd.DataFrame(sheet_id.worksheet('Detalle_Ventas').get_all_records())
df_productos = pd.DataFrame(sheet_id.worksheet('Productos').get_all_records())
df_gastos = pd.DataFrame(sheet_id.worksheet('Gastos').get_all_records())

print("🧹 2/2: Curando los datos (forzando formato numérico)...")
df_detalle['cantidad'] = pd.to_numeric(df_detalle['cantidad'], errors='coerce').fillna(0)
df_detalle['precio_compra_aplicado'] = pd.to_numeric(df_detalle['precio_compra_aplicado'], errors='coerce').fillna(0)
df_detalle['precio_venta_aplicado'] = pd.to_numeric(df_detalle['precio_venta_aplicado'], errors='coerce').fillna(0)

print("✅ Datos extraídos y limpios en memoria.")

📥 1/2: Extrayendo tablas desde Google Sheets...
🧹 2/2: Curando los datos (forzando formato numérico)...
✅ Datos extraídos y limpios en memoria.


In [3]:
print("📦 Calculando el Stock Actual en memoria...")

# Consulta SQL para calcular el inventario usando el stock_inicial
query_inventario = """
    SELECT 
        p.id_producto,
        p.nombre,
        p.stock_inicial,
        COALESCE(SUM(d.cantidad), 0) AS total_vendido,
        (p.stock_inicial - COALESCE(SUM(d.cantidad), 0)) AS stock_actual
    FROM df_productos AS p
    LEFT JOIN df_detalle AS d ON p.id_producto = d.id_producto
    GROUP BY p.id_producto, p.nombre, p.stock_inicial
"""

# Ejecutar y limpiar nulos
df_inventario = duckdb.query(query_inventario).to_df()
df_inventario = df_inventario.fillna("")

print("✅ Inventario calculado. Muestra de los datos:")
display(df_inventario.head())

📦 Calculando el Stock Actual en memoria...
✅ Inventario calculado. Muestra de los datos:


,id_producto,nombre,stock_inicial,total_vendido,stock_actual
0,9902f327,Coca Cola 600ml,12,5.0,7.0


In [4]:
print("📤 Subiendo tabla de Inventario a Google Sheets...")

# Gestionar la pestaña destino para el Inventario
try:
    hoja_inv = sheet_id.add_worksheet(title="BI_Inventario", rows="1000", cols="10")
    print("✨ Pestaña 'BI_Inventario' creada desde cero.")
except Exception:
    hoja_inv = sheet_id.worksheet("BI_Inventario")
    hoja_inv.clear()
    print("♻️ Pestaña 'BI_Inventario' limpiada con éxito.")

# Formatear para Google Sheets y subir
datos_inv = [df_inventario.columns.values.tolist()] + df_inventario.values.tolist()
hoja_inv.update(range_name='A1', values=datos_inv)

print("🚀 ¡Todo listo! Datos de ventas e inventario actualizados en la nube.")

📤 Subiendo tabla de Inventario a Google Sheets...
✨ Pestaña 'BI_Inventario' creada desde cero.
🚀 ¡Todo listo! Datos de ventas e inventario actualizados en la nube.
